# Week 5–6 — PolicyNet training and checkpoint audit

Canonical run: `policy-v2-mcts-strength8-seed20261109`. This notebook calls the tested modules/script; it does not contain a second training loop.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs/policy_train_v2_mcts_strength.json'
DATASET_DIR = PROJECT_ROOT / 'data/processed_tensors/w56_dataset_v2_mcts_strength8_2000_balanced'
OUTPUT_DIR = PROJECT_ROOT / 'models/checkpoints/model-v2-mcts-strength8'
RUN_SMOKE = False  # Set True to execute the canonical two-batch CPU smoke run.
print({'config': str(CONFIG_PATH), 'dataset': str(DATASET_DIR), 'output': str(OUTPUT_DIR)})

In [ ]:
from core.ml.data_pipeline import load_dataset_split, sha256_file
from core.ml.architectures import PolicyNet, PolicyNetConfig, masked_policy_loss
from scripts.train_policy import load_training_config, validate_dataset_contract

config = load_training_config(CONFIG_PATH)
model_config = PolicyNetConfig.from_mapping(config['model'])
manifest = validate_dataset_contract(DATASET_DIR, model_config, config['dataset'])
assert sha256_file(DATASET_DIR / 'dataset_manifest.json') == config['dataset']['manifest_sha256']
print({'run_id': config['run_id'], 'dataset_version': manifest['dataset_version'], 'manifest_sha256': config['dataset']['manifest_sha256']})

In [ ]:
arrays = load_dataset_split(DATASET_DIR, 'train', verify_checksum=True)
x = torch.from_numpy(arrays['x'][:4])
legal_mask = torch.from_numpy(arrays['legal_mask'][:4])
target = torch.from_numpy(arrays['target'][:4])
model = PolicyNet(model_config)
logits = model(x)
loss = masked_policy_loss(logits, legal_mask, target)
loss.backward()
print({'x': tuple(x.shape), 'logits': tuple(logits.shape), 'loss': float(loss.detach()), 'finite_gradients': all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)})

In [ ]:
command = [sys.executable, str(PROJECT_ROOT / 'scripts/train_policy.py'), '--config', str(CONFIG_PATH), '--dataset', str(DATASET_DIR), '--output', str(OUTPUT_DIR), '--device', 'cpu', '--smoke']
print(' '.join(command))
if RUN_SMOKE:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [ ]:
history_path = OUTPUT_DIR / 'history.csv'
summary_path = OUTPUT_DIR / 'run_summary.json'
if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history)
    history.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', title='PolicyNet learning curve')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('No local history yet; run the canonical command above.')
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    display({'best_epoch': summary['best_epoch'], 'best_val_loss': summary['best_val_loss'], 'test_metrics': summary['test_metrics'], 'parameter_count': summary['parameter_count']})